# 07 — LightGBM Ranker — MerRec

## Mục tiêu

Notebook này xây dựng **LightGBM Ranker (LambdaMART / LambdaRank)** để xếp hạng lại candidate sản phẩm sau tầng Retrieval.

Pipeline:

**Content-Based + CoVisitation + ALS + GRU4Rec + Two-Tower**
→ Candidate Fusion
→ Feature Engineering
→ **LightGBM Ranker**
→ Top-K Recommendation

Nếu dataset có thêm `dcn_score`, LightGBM sẽ tự động dùng `dcn_score` như một feature bổ sung.

### Input

Notebook mặc định đọc:

- `D:\MerRec\training\dcn\dataset\dcn_train.parquet`
- `D:\MerRec\training\dcn\dataset\dcn_val.parquet`
- `D:\MerRec\training\dcn\dataset\dcn_test.parquet`

Mỗi dòng là một `(case_id, candidate_item_id)` với `label`:

- `label = 1`: candidate là next-item thật.
- `label = 0`: candidate âm.

### Output

- `lightgbm_ranker.txt`
- `feature_importance.csv`
- `train_history.json`
- `val_metrics.json`
- `test_metrics.json`
- `test_ranked_topk.parquet`

## 1. Cài thư viện

In [ ]:
# Nếu máy chưa có LightGBM thì chạy 1 lần:
# !pip install lightgbm polars pyarrow pandas numpy scikit-learn

## 2. Imports + cấu hình

In [ ]:
from pathlib import Path
import json
import math
import hashlib
import os
import warnings

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb

from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

SEED = 42

PROJECT_ROOT = Path(r"D:\MerRec")

DCN_DATASET_DIR = (
    PROJECT_ROOT
    / "training"
    / "dcn"
    / "dataset"
)

TRAIN_PATH = DCN_DATASET_DIR / "dcn_train.parquet"
VAL_PATH = DCN_DATASET_DIR / "dcn_val.parquet"
TEST_PATH = DCN_DATASET_DIR / "dcn_test.parquet"

LGB_ROOT = (
    PROJECT_ROOT
    / "training"
    / "lightgbm_ranker"
)

MODEL_DIR = LGB_ROOT / "model"
METRIC_DIR = LGB_ROOT / "metrics"
OUTPUT_DIR = LGB_ROOT / "outputs"

for d in [LGB_ROOT, MODEL_DIR, METRIC_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("LightGBM:", lgb.__version__)
print("TRAIN:", TRAIN_PATH)
print("VAL  :", VAL_PATH)
print("TEST :", TEST_PATH)
print("OUT  :", LGB_ROOT)

## 3. Load candidate datasets

Điều kiện quan trọng với Learning-to-Rank:

- các row cùng `case_id` phải nằm liền nhau;
- `group` phải chứa số candidate của từng `case_id`;
- `label` là relevance label.

Notebook sẽ sort theo `case_id` trước khi train.

In [ ]:
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Không tìm thấy {p}. "
            "Hãy chạy Candidate Fusion/DCN data preparation trước."
        )

train_pl = pl.read_parquet(TRAIN_PATH)
val_pl = pl.read_parquet(VAL_PATH)
test_pl = pl.read_parquet(TEST_PATH)

print("TRAIN:", train_pl.shape)
print("VAL  :", val_pl.shape)
print("TEST :", test_pl.shape)

print("\nTRAIN columns:")
print(train_pl.columns)

## 4. Kiểm tra Ranking Dataset

Ta kiểm tra:

- số `case_id`;
- số positive;
- số case có positive;
- candidate trung bình/case;
- Candidate Recall ceiling.

In [ ]:
def dataset_summary(df, name):
    cases = df["case_id"].n_unique()
    positives = int(df["label"].sum())

    positive_cases = (
        df
        .filter(pl.col("label") > 0)
        ["case_id"]
        .n_unique()
    )

    avg_candidates = (
        df.height / cases
        if cases > 0
        else 0
    )

    candidate_recall = (
        positive_cases / cases
        if cases > 0
        else 0
    )

    print("=" * 65)
    print(name)
    print("rows              :", f"{df.height:,}")
    print("cases             :", f"{cases:,}")
    print("positives         :", f"{positives:,}")
    print("positive cases    :", f"{positive_cases:,}")
    print("avg candidates    :", round(avg_candidates, 2))
    print("candidate recall  :", round(candidate_recall, 6))


dataset_summary(train_pl, "TRAIN")
dataset_summary(val_pl, "VAL")
dataset_summary(test_pl, "TEST")

## 5. Feature selection

Không được đưa vào model:

- `case_id`
- `target_item_id`
- `label`

`candidate_item_id`, `user_id`, brand/category... được hash thành categorical integer feature.

Retrieval scores/ranks, price, popularity, RRF, source_count... giữ dạng numerical.

Nếu có `dcn_score`, feature này được giữ tự động.

In [ ]:
ID_COLUMNS = {
    "case_id",
    "target_item_id",
    "label",
}

CATEGORICAL_COLUMNS = [
    c for c in [
        "user_id",
        "candidate_item_id",
        "candidate_product_id",
        "candidate_brand",
        "candidate_category0",
        "candidate_category1",
        "candidate_category2",
        "candidate_condition",
        "candidate_shipper",
        "last_item_id",
        "last_product_id",
        "last_event_id",
        "last_brand",
        "last_category0",
    ]
    if c in train_pl.columns
]

# Mọi column còn lại ngoài ID + categorical sẽ dùng như numerical.
NUMERICAL_COLUMNS = [
    c for c in train_pl.columns
    if c not in ID_COLUMNS
    and c not in CATEGORICAL_COLUMNS
]

print("Categorical features:", len(CATEGORICAL_COLUMNS))
print(CATEGORICAL_COLUMNS)

print("\nNumerical features:", len(NUMERICAL_COLUMNS))
print(NUMERICAL_COLUMNS)

if "dcn_score" in NUMERICAL_COLUMNS:
    print("\n✅ dcn_score sẽ được dùng làm feature.")
else:
    print("\nℹ️ Không có dcn_score; LightGBM vẫn train độc lập trên candidate features.")

## 6. Hash categorical features

Hash giúp tránh tạo mapping khổng lồ cho hàng triệu user/item.

Các giá trị categorical được hash về `Int32`.
Cùng một seed được dùng cho TRAIN/VAL/TEST.

In [ ]:
HASH_BUCKETS = {
    "user_id": 300_000,
    "candidate_item_id": 700_000,
    "candidate_product_id": 400_000,
    "candidate_brand": 80_000,
    "candidate_category0": 20_000,
    "candidate_category1": 40_000,
    "candidate_category2": 80_000,
    "candidate_condition": 2_048,
    "candidate_shipper": 4_096,
    "last_item_id": 700_000,
    "last_product_id": 400_000,
    "last_event_id": 256,
    "last_brand": 80_000,
    "last_category0": 20_000,
}

def add_hashed_categoricals(df):
    exprs = []

    for col in CATEGORICAL_COLUMNS:
        buckets = HASH_BUCKETS.get(col, 100_000)

        exprs.append(
            (
                pl.col(col)
                .cast(pl.Utf8)
                .fill_null("unknown")
                .hash(seed=SEED)
                % buckets
            )
            .cast(pl.Int32)
            .alias(col)
        )

    if exprs:
        df = df.with_columns(exprs)

    return df


train_pl = add_hashed_categoricals(train_pl)
val_pl = add_hashed_categoricals(val_pl)
test_pl = add_hashed_categoricals(test_pl)

print("✅ categorical hashing done")

## 7. Làm sạch numerical features

- `NaN`, `+inf`, `-inf` được xử lý.
- Rank `9999` vẫn được giữ vì nó biểu diễn model retrieval không trả candidate đó.
- Không standardize vì tree model không cần chuẩn hóa thang đo.

In [ ]:
def clean_numerical(df):
    exprs = []

    for col in NUMERICAL_COLUMNS:
        exprs.append(
            pl.col(col)
            .cast(pl.Float32, strict=False)
            .fill_nan(0.0)
            .fill_null(0.0)
            .alias(col)
        )

    return df.with_columns(exprs)


train_pl = clean_numerical(train_pl)
val_pl = clean_numerical(val_pl)
test_pl = clean_numerical(test_pl)

print("✅ numerical cleaning done")

## 8. Sort theo `case_id` và tạo group sizes

LightGBM Ranker cần `group`:

Ví dụ:

- case 1 có 150 candidate
- case 2 có 180 candidate
- case 3 có 120 candidate

thì:

`group = [150, 180, 120]`

In [ ]:
FEATURE_COLUMNS = (
    CATEGORICAL_COLUMNS
    + NUMERICAL_COLUMNS
)

def prepare_rank_data(df):
    # Sort để các candidate của cùng một query nằm cạnh nhau.
    df = df.sort(["case_id", "candidate_item_id"])

    group_df = (
        df
        .group_by("case_id", maintain_order=True)
        .len()
        .rename({"len": "group_size"})
    )

    groups = (
        group_df["group_size"]
        .to_numpy()
        .astype(np.int32)
    )

    X = (
        df
        .select(FEATURE_COLUMNS)
        .to_pandas()
    )

    y = (
        df["label"]
        .to_numpy()
        .astype(np.int32)
    )

    case_ids = df["case_id"].to_numpy()
    candidate_ids = df["candidate_item_id"].to_numpy()
    target_ids = df["target_item_id"].to_numpy()

    return df, X, y, groups, case_ids, candidate_ids, target_ids


(
    train_sorted,
    X_train,
    y_train,
    group_train,
    train_case_ids,
    train_candidate_ids,
    train_target_ids,
) = prepare_rank_data(train_pl)

(
    val_sorted,
    X_val,
    y_val,
    group_val,
    val_case_ids,
    val_candidate_ids,
    val_target_ids,
) = prepare_rank_data(val_pl)

(
    test_sorted,
    X_test,
    y_test,
    group_test,
    test_case_ids,
    test_candidate_ids,
    test_target_ids,
) = prepare_rank_data(test_pl)

print("X_train:", X_train.shape)
print("groups train:", len(group_train), "sum =", int(group_train.sum()))
print("groups val  :", len(group_val), "sum =", int(group_val.sum()))
print("groups test :", len(group_test), "sum =", int(group_test.sum()))

assert int(group_train.sum()) == len(X_train)
assert int(group_val.sum()) == len(X_val)
assert int(group_test.sum()) == len(X_test)

## 9. LightGBM Ranker

Sử dụng:

- Objective: `lambdarank`
- Metric: `ndcg`
- Early stopping theo validation
- CPU training

`eval_at = [10, 20, 50, 100]`

In [ ]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",

    boosting_type="gbdt",

    n_estimators=2000,
    learning_rate=0.03,

    num_leaves=63,
    max_depth=-1,

    min_child_samples=50,
    min_split_gain=0.0,

    subsample=0.85,
    subsample_freq=1,

    colsample_bytree=0.85,

    reg_alpha=0.05,
    reg_lambda=0.20,

    max_bin=255,

    n_jobs=-1,
    random_state=SEED,

    verbosity=-1,
)

print(ranker)

## 10. Train

Best iteration được chọn bằng Validation NDCG.

Categorical features được truyền trực tiếp cho LightGBM.

In [ ]:
callbacks = [
    lgb.early_stopping(
        stopping_rounds=100,
        first_metric_only=False,
        verbose=True,
    ),
    lgb.log_evaluation(period=25),
]

ranker.fit(
    X_train,
    y_train,

    group=group_train,

    eval_set=[
        (X_val, y_val)
    ],

    eval_group=[
        group_val
    ],

    eval_at=[
        10,
        20,
        50,
        100,
    ],

    categorical_feature=CATEGORICAL_COLUMNS,

    callbacks=callbacks,
)

print("\nBest iteration:", ranker.best_iteration_)
print("Best scores:")
print(json.dumps(ranker.best_score_, indent=2))

## 11. Save model

In [ ]:
MODEL_PATH = (
    MODEL_DIR
    / "lightgbm_ranker.txt"
)

ranker.booster_.save_model(
    str(MODEL_PATH)
)

print("✅ Saved:", MODEL_PATH)

## 12. Ranking metrics

Với next-item, mỗi case có tối đa một positive target.

Đánh giá:

- Recall@10 / 20 / 50 / 100
- NDCG@10 / 20 / 50 / 100
- MRR@20
- AUC (chỉ tham khảo)

In [ ]:
def ranking_metrics(
    case_ids,
    candidate_ids,
    labels,
    scores,
    ks=(10, 20, 50, 100),
):
    df = pd.DataFrame({
        "case_id": case_ids,
        "candidate_item_id": candidate_ids,
        "label": labels,
        "score": scores,
    })

    recall = {k: [] for k in ks}
    ndcg = {k: [] for k in ks}
    mrr20 = []

    all_cases = 0
    positive_cases = 0

    for _, g in df.groupby("case_id", sort=False):
        all_cases += 1

        if g["label"].sum() <= 0:
            continue

        positive_cases += 1

        g = g.sort_values(
            "score",
            ascending=False,
        )

        ranked_labels = g["label"].to_numpy()

        pos = np.where(
            ranked_labels > 0
        )[0]

        if len(pos) == 0:
            continue

        rank = int(pos[0]) + 1

        for k in ks:
            hit = rank <= k

            recall[k].append(
                float(hit)
            )

            ndcg[k].append(
                1.0 / math.log2(rank + 1)
                if hit
                else 0.0
            )

        mrr20.append(
            1.0 / rank
            if rank <= 20
            else 0.0
        )

    out = {
        "all_cases": int(all_cases),
        "positive_cases": int(positive_cases),
        "candidate_recall": (
            positive_cases / all_cases
            if all_cases > 0
            else 0.0
        ),
    }

    for k in ks:
        out[f"Recall@{k}"] = (
            float(np.mean(recall[k]))
            if recall[k]
            else 0.0
        )

        out[f"NDCG@{k}"] = (
            float(np.mean(ndcg[k]))
            if ndcg[k]
            else 0.0
        )

    out["MRR@20"] = (
        float(np.mean(mrr20))
        if mrr20
        else 0.0
    )

    return out

## 13. Validation metrics

In [ ]:
val_scores = ranker.predict(
    X_val,
    num_iteration=ranker.best_iteration_,
)

val_metrics = ranking_metrics(
    val_case_ids,
    val_candidate_ids,
    y_val,
    val_scores,
)

try:
    val_metrics["AUC"] = float(
        roc_auc_score(
            y_val,
            val_scores,
        )
    )
except Exception:
    val_metrics["AUC"] = None

print(
    json.dumps(
        val_metrics,
        indent=2,
    )
)

with open(
    METRIC_DIR / "val_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        val_metrics,
        f,
        indent=2,
    )

## 14. Final TEST

TEST chỉ đánh giá sau khi model đã chọn xong trên validation.

In [ ]:
test_scores = ranker.predict(
    X_test,
    num_iteration=ranker.best_iteration_,
)

test_metrics = ranking_metrics(
    test_case_ids,
    test_candidate_ids,
    y_test,
    test_scores,
)

try:
    test_metrics["AUC"] = float(
        roc_auc_score(
            y_test,
            test_scores,
        )
    )
except Exception:
    test_metrics["AUC"] = None

print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

with open(
    METRIC_DIR / "test_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        test_metrics,
        f,
        indent=2,
    )

## 15. Feature Importance

Dùng gain importance để xem feature nào đóng góp nhiều nhất.

In [ ]:
importance = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance_gain": ranker.booster_.feature_importance(
        importance_type="gain"
    ),
    "importance_split": ranker.booster_.feature_importance(
        importance_type="split"
    ),
})

importance = importance.sort_values(
    "importance_gain",
    ascending=False,
).reset_index(drop=True)

IMPORTANCE_PATH = (
    OUTPUT_DIR
    / "feature_importance.csv"
)

importance.to_csv(
    IMPORTANCE_PATH,
    index=False,
)

print(importance.head(30))
print("\nSaved:", IMPORTANCE_PATH)

## 16. Xuất Top-K recommendation trên TEST

Lưu ranking cuối theo score của LightGBM.

Mặc định lưu Top-100 mỗi `case_id`.

In [ ]:
TOPK_FINAL = 100

ranked_test = (
    test_sorted
    .with_columns(
        pl.Series(
            "lightgbm_score",
            test_scores.astype(np.float32),
        )
    )
    .sort(
        ["case_id", "lightgbm_score"],
        descending=[False, True],
    )
    .group_by(
        "case_id",
        maintain_order=True,
    )
    .head(TOPK_FINAL)
    .with_columns(
        pl.int_range(
            1,
            pl.len() + 1
        )
        .over("case_id")
        .cast(pl.Int32)
        .alias("final_rank")
    )
)

TOPK_PATH = (
    OUTPUT_DIR
    / "test_ranked_top100.parquet"
)

ranked_test.write_parquet(
    TOPK_PATH,
    compression="zstd",
)

print(
    "Saved:",
    TOPK_PATH
)

print(
    "rows:",
    f"{ranked_test.height:,}"
)

print(
    "cases:",
    f"{ranked_test['case_id'].n_unique():,}"
)

## 17. Save summary

In [ ]:
summary = {
    "model": "LightGBM Ranker",
    "objective": "lambdarank",
    "best_iteration": int(
        ranker.best_iteration_
        if ranker.best_iteration_
        else ranker.n_estimators
    ),
    "features": FEATURE_COLUMNS,
    "categorical_features": CATEGORICAL_COLUMNS,
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "model_path": str(MODEL_PATH),
    "topk_path": str(TOPK_PATH),
}

SUMMARY_PATH = (
    METRIC_DIR
    / "summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    )
)

# Kết quả cuối

Pipeline của đề tài có thể trình bày:

1. **Retrieval models** tạo candidate.
2. **DCN** học tương tác phức tạp giữa user, item, behavior và retrieval signals.
3. **LightGBM Ranker** dùng LambdaRank để tối ưu trực tiếp thứ tự candidate.
4. Xuất danh sách **Top-K sản phẩm gợi ý cuối cùng**.

> Nếu muốn LightGBM đứng sau DCN đúng nghĩa, hãy thêm `dcn_score` vào các file `dcn_train/val/test.parquet`. Notebook này sẽ tự động dùng feature đó.